In [1]:
from pathlib import Path
from urllib.parse import urlparse
import json
import boto3
import pandas as pd

PROJECT_ROOT = Path.cwd()
config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")

def load_saved_split(name):
    """Read the exact S3 file version recorded in our configuration."""
    file_info = project_config["s3_data"][name]
    location = urlparse(file_info["uri"])

    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=file_info["version_id"]
    )

    try:
        return pd.read_csv(
            response["Body"],
            dtype={"ticket_id": "string"},
            low_memory=False
        )
    finally:
        response["Body"].close()

train_df = load_saved_split("train")
validation_df = load_saved_split("validation")

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Columns:", train_df.columns.tolist())

Training rows: 17710
Validation rows: 4428
Columns: ['ticket_id', 'subject', 'body', 'language', 'priority', 'type', 'queue']


In [2]:
def engineer_features(df):
    result = df.copy()

    # Prepare the ticket text.
    subject = result["subject"].fillna("").astype(str).str.strip()
    body = result["body"].fillna("").astype(str).str.strip()

    result["combined_text"] = (subject + " " + body).str.strip()

    # Structural features: whether a subject exists and text lengths.
    result["has_subject"] = subject.ne("").astype(int)

    result["subject_char_count"] = subject.str.len()
    result["body_char_count"] = body.str.len()
    result["text_char_count"] = result["combined_text"].str.len()

    result["subject_word_count"] = subject.str.split().str.len()
    result["body_word_count"] = body.str.split().str.len()
    result["text_word_count"] = (
        result["combined_text"].str.split().str.len()
    )

    return result


train_features = engineer_features(train_df)
validation_features = engineer_features(validation_df)

structural_columns = [
    "has_subject",
    "subject_char_count",
    "body_char_count",
    "text_char_count",
    "subject_word_count",
    "body_word_count",
    "text_word_count",
]

# Confirm feature creation preserved the tickets and produced usable text.
for name, original, features in [
    ("Training", train_df, train_features),
    ("Validation", validation_df, validation_features),
]:
    assert features["ticket_id"].equals(original["ticket_id"])
    assert features["combined_text"].str.len().gt(0).all()
    assert features[structural_columns].notna().all().all()
    print(f"{name}: {len(features):,} tickets — features verified")

display(train_features[structural_columns].head())

Training: 17,710 tickets — features verified
Validation: 4,428 tickets — features verified


,has_subject,subject_char_count,body_char_count,text_char_count,subject_word_count,body_word_count,text_word_count
0,1,51,312,364,7,48,55
1,1,38,446,485,4,54,58
2,1,35,519,555,4,75,79
3,1,21,270,292,3,40,43
4,0,0,776,776,0,89,89


In [3]:
TEXT_COLUMN = "combined_text"
CATEGORICAL_COLUMNS = ["language"]
NUMERIC_COLUMNS = structural_columns.copy()
TARGET_COLUMN = "queue"

MODEL_INPUT_COLUMNS = [
    TEXT_COLUMN,
    *CATEGORICAL_COLUMNS,
    *NUMERIC_COLUMNS,
]

for features in [train_features, validation_features]:
    features["language"] = (
        features["language"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("", "unknown")
    )

# IDs identify records; queue is the label. Neither is a model input.
assert "ticket_id" not in MODEL_INPUT_COLUMNS
assert TARGET_COLUMN not in MODEL_INPUT_COLUMNS

print("Text input:", TEXT_COLUMN)
print("Categorical inputs:", CATEGORICAL_COLUMNS)
print("Numeric inputs:", NUMERIC_COLUMNS)
print("Prediction target:", TARGET_COLUMN)
print("Training queues:", train_features[TARGET_COLUMN].nunique())

Text input: combined_text
Categorical inputs: ['language']
Numeric inputs: ['has_subject', 'subject_char_count', 'body_char_count', 'text_char_count', 'subject_word_count', 'body_word_count', 'text_word_count']
Prediction target: queue
Training queues: 10


In [4]:
# Connect to the AWS services.
sagemaker_client = aws_session.client("sagemaker")
featurestore_runtime = aws_session.client("sagemaker-featurestore-runtime")
iam_client = aws_session.client("iam")

# Use the existing AWS Academy execution role.
EXECUTION_ROLE_ARN = iam_client.get_role(
    RoleName="LabRole"
)["Role"]["Arn"]

snapshot_id = project_config["dataset_sha256"][:12]

FEATURE_GROUP_NAME = f"aai540-group3-tickets-{snapshot_id}-v1"
FEATURE_STORE_S3_URI = (
    f"s3://{PROJECT_BUCKET}/feature-store/"
)

# Define the columns and their storage types.
# Metadata and labels are stored alongside features,
# but only MODEL_INPUT_COLUMNS will enter the model.
feature_definitions = [
    {"FeatureName": "ticket_id", "FeatureType": "String"},
    {"FeatureName": "event_time", "FeatureType": "Fractional"},
    {"FeatureName": "split", "FeatureType": "String"},
    {"FeatureName": "combined_text", "FeatureType": "String"},
    {"FeatureName": "language", "FeatureType": "String"},
    {"FeatureName": "queue", "FeatureType": "String"},
]

feature_definitions += [
    {"FeatureName": column, "FeatureType": "Integral"}
    for column in NUMERIC_COLUMNS
]

print("Execution role:", EXECUTION_ROLE_ARN)
print("Feature group:", FEATURE_GROUP_NAME)
print("Offline storage:", FEATURE_STORE_S3_URI)
print("Registered column definitions:", len(feature_definitions))

Execution role: arn:aws:iam::390568313988:role/LabRole
Feature group: aai540-group3-tickets-c142eef90e9f-v1
Offline storage: s3://aai540-group3-tickets-390568313988-us-east-1/feature-store/
Registered column definitions: 13


In [5]:
from botocore.exceptions import ClientError

try:
    feature_group_info = sagemaker_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )
    print("Feature group already exists.")

except ClientError as error:
    if error.response["Error"]["Code"] != "ResourceNotFound":
        raise

    sagemaker_client.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierFeatureName="ticket_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=feature_definitions,
        OfflineStoreConfig={
            "S3StorageConfig": {
                "S3Uri": FEATURE_STORE_S3_URI
            },
            "DisableGlueTableCreation": False
        },
        RoleArn=EXECUTION_ROLE_ARN,
        Description=(
            "Multilingual ticket text and structural features. "
            "Includes split membership and queue labels."
        )
    )
    print("Feature group creation requested.")

    feature_group_info = sagemaker_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

print("Feature group:", FEATURE_GROUP_NAME)
print("Status:", feature_group_info["FeatureGroupStatus"])

if feature_group_info.get("FailureReason"):
    print("Failure reason:", feature_group_info["FailureReason"])

Feature group creation requested.
Feature group: aai540-group3-tickets-c142eef90e9f-v1
Status: Creating


In [6]:
import time

# Wait up to one minute.
for attempt in range(12):
    feature_group_info = sagemaker_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )
    status = feature_group_info["FeatureGroupStatus"]

    if status == "Created":
        break

    if status != "Creating":
        raise RuntimeError(
            f"Feature group status: {status}. "
            f"{feature_group_info.get('FailureReason', '')}"
        )

    time.sleep(5)

else:
    raise TimeoutError(
        "AWS is still creating the feature group. "
        "Wait a minute, then rerun this cell."
    )

# Save the details so another notebook or session can reuse them.
project_config.setdefault("feature_store", {}).update({
    "feature_group_name": FEATURE_GROUP_NAME,
    "feature_group_arn": feature_group_info["FeatureGroupArn"],
    "execution_role_arn": EXECUTION_ROLE_ARN,
    "offline_s3_uri": FEATURE_STORE_S3_URI,
    "model_input_columns": MODEL_INPUT_COLUMNS,
    "target_column": TARGET_COLUMN,
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Feature group status:", status)
print(
    "Offline store status:",
    feature_group_info.get("OfflineStoreStatus", {})
    .get("Status", "Not reported")
)
print("Feature Store configuration saved.")

Feature group status: Created
Offline store status: Not reported
Feature Store configuration saved.


In [7]:
import time

# Keep one timestamp for this feature snapshot across reruns.
# This is the feature preparation time, not the original ticket date.
feature_event_time = project_config["feature_store"].setdefault(
    "feature_event_time", time.time()
)

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

stored_columns = ["ticket_id", *MODEL_INPUT_COLUMNS, TARGET_COLUMN]

feature_records_df = pd.concat(
    [
        train_features[stored_columns].assign(split="train"),
        validation_features[stored_columns].assign(split="validation"),
    ],
    ignore_index=True
)

feature_records_df["event_time"] = feature_event_time

# Match the feature group's schema.
store_columns = [
    definition["FeatureName"]
    for definition in feature_definitions
]
feature_records_df = feature_records_df[store_columns]

assert feature_records_df["ticket_id"].is_unique
assert feature_records_df.notna().all().all()

def make_feature_record(row):
    return [
        {"FeatureName": column, "ValueAsString": str(row[column])}
        for column in store_columns
    ]

# Send one record before uploading the remaining records.
sample_row = feature_records_df.iloc[0]

featurestore_runtime.put_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    Record=make_feature_record(sample_row),
    TargetStores=["OfflineStore"]
)

sample_ticket_id = str(sample_row["ticket_id"])

print("Records prepared:", len(feature_records_df))
print(feature_records_df["split"].value_counts())
print("Sample record accepted by Feature Store.")

Records prepared: 22138
split
train         17710
validation     4428
Name: count, dtype: int64
Sample record accepted by Feature Store.


In [8]:
from concurrent.futures import ThreadPoolExecutor
from botocore.config import Config

# Retry temporary AWS errors automatically.
featurestore_runtime = aws_session.client(
    "sagemaker-featurestore-runtime",
    config=Config(
        retries={"mode": "standard", "max_attempts": 10},
        max_pool_connections=10
    )
)

# Keep upload progress separate for each feature snapshot.
progress_dir = PROJECT_ROOT / "data" / "feature_store"
progress_dir.mkdir(parents=True, exist_ok=True)

progress_file = progress_dir / (
    f"{FEATURE_GROUP_NAME}-{int(feature_event_time * 1_000_000)}.txt"
)

uploaded_ids = (
    set(progress_file.read_text().splitlines())
    if progress_file.exists()
    else set()
)

# The sample record was already accepted.
with progress_file.open("a", encoding="utf-8") as log:
    if sample_ticket_id not in uploaded_ids:
        log.write(sample_ticket_id + "\n")
        uploaded_ids.add(sample_ticket_id)

pending_records = feature_records_df.loc[
    ~feature_records_df["ticket_id"].isin(uploaded_ids)
].to_dict("records")

def upload_feature_record(row):
    ticket_id = str(row["ticket_id"])
    try:
        featurestore_runtime.put_record(
            FeatureGroupName=FEATURE_GROUP_NAME,
            Record=make_feature_record(row),
            TargetStores=["OfflineStore"]
        )
        return ticket_id, None
    except Exception as error:
        return ticket_id, str(error)

print(f"Records remaining to upload: {len(pending_records):,}", flush=True)
failed_uploads = []

with progress_file.open("a", encoding="utf-8") as log:
    with ThreadPoolExecutor(max_workers=8) as executor:
        for ticket_id, error in executor.map(
            upload_feature_record, pending_records
        ):
            if error is not None:
                failed_uploads.append((ticket_id, error))
                continue

            uploaded_ids.add(ticket_id)
            log.write(ticket_id + "\n")
            log.flush()

            if len(uploaded_ids) % 1000 == 0:
                print(
                    f"Accepted: {len(uploaded_ids):,} / "
                    f"{len(feature_records_df):,}",
                    flush=True
                )

print(f"Upload failures: {len(failed_uploads)}")

if failed_uploads:
    print("First error:", failed_uploads[0][1])
    raise RuntimeError(
        "Some uploads failed. Share the error before continuing."
    )

assert set(feature_records_df["ticket_id"]).issubset(uploaded_ids)
print(f"All {len(feature_records_df):,} feature records accepted.")

Records remaining to upload: 22,137
Accepted: 1,000 / 22,138
Accepted: 2,000 / 22,138
Accepted: 3,000 / 22,138
Accepted: 4,000 / 22,138
Accepted: 5,000 / 22,138
Accepted: 6,000 / 22,138
Accepted: 7,000 / 22,138
Accepted: 8,000 / 22,138
Accepted: 9,000 / 22,138
Accepted: 10,000 / 22,138
Accepted: 11,000 / 22,138
Accepted: 12,000 / 22,138
Accepted: 13,000 / 22,138
Accepted: 14,000 / 22,138
Accepted: 15,000 / 22,138
Accepted: 16,000 / 22,138
Accepted: 17,000 / 22,138
Accepted: 18,000 / 22,138
Accepted: 19,000 / 22,138
Accepted: 20,000 / 22,138
Accepted: 21,000 / 22,138
Accepted: 22,000 / 22,138
Upload failures: 0
All 22,138 feature records accepted.


In [10]:
athena_client = aws_session.client("athena")

# Discover the table AWS created for this feature group.
feature_group_info = sagemaker_client.describe_feature_group(
    FeatureGroupName=FEATURE_GROUP_NAME
)
catalog_info = feature_group_info[
    "OfflineStoreConfig"
]["DataCatalogConfig"]

FEATURE_DATABASE = catalog_info["Database"]
FEATURE_TABLE = catalog_info["TableName"]

print("Database:", FEATURE_DATABASE)
print("Table:", FEATURE_TABLE)

# Count unique tickets in this feature snapshot.
# DISTINCT prevents any retried uploads from inflating the counts.
count_sql = f"""
SELECT
    split,
    COUNT(DISTINCT ticket_id) AS ticket_count
FROM "{FEATURE_DATABASE}"."{FEATURE_TABLE}"
WHERE is_deleted = false
  AND event_time = CAST({feature_event_time!r} AS DOUBLE)
GROUP BY split
ORDER BY split
"""

query_id = athena_client.start_query_execution(
    QueryString=count_sql,
    QueryExecutionContext={
        "Database": FEATURE_DATABASE,
        "Catalog": "AwsDataCatalog"
    },
    ResultConfiguration={
        "OutputLocation": project_config["athena"]["query_output"]
    },
    WorkGroup=project_config["athena"]["workgroup"]
)["QueryExecutionId"]

print("Query ID:", query_id)

for attempt in range(30):
    query_status = athena_client.get_query_execution(
        QueryExecutionId=query_id
    )["QueryExecution"]["Status"]

    state = query_status["State"]
    if state == "SUCCEEDED":
        break
    if state in {"FAILED", "CANCELLED"}:
        raise RuntimeError(query_status.get("StateChangeReason", state))
    time.sleep(2)
else:
    raise TimeoutError(f"Query is still running: {query_id}")

rows = athena_client.get_query_results(
    QueryExecutionId=query_id
)["ResultSet"]["Rows"]

actual_counts = {
    row["Data"][0]["VarCharValue"]:
        int(row["Data"][1]["VarCharValue"])
    for row in rows[1:]
}

expected_counts = {"train": 17710, "validation": 4428}
print("Expected:", expected_counts)
print("Available:", actual_counts)

if actual_counts == expected_counts:
    print("Verified: both feature splits are available in Athena.")
else:
    print("Counts do not match yet. Share this output before continuing.")

Database: sagemaker_featurestore
Table: aai540_group3_tickets_c142eef90e9f_v1_1791068947
Query ID: 6b14f7a0-ba17-411d-b21b-b38a49fc40d5
Expected: {'train': 17710, 'validation': 4428}
Available: {'train': 17710, 'validation': 4428}
Verified: both feature splits are available in Athena.


In [11]:
# Select one copy of each ticket if any upload was retried.
selected_columns = ", ".join(f'"{c}"' for c in store_columns)

feature_snapshot_sql = f"""
WITH ranked_records AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY ticket_id
            ORDER BY api_invocation_time DESC, write_time DESC
        ) AS row_num
    FROM "{FEATURE_DATABASE}"."{FEATURE_TABLE}"
    WHERE event_time = CAST({feature_event_time!r} AS DOUBLE)
)
SELECT {selected_columns}
FROM ranked_records
WHERE row_num = 1 AND is_deleted = false
"""

feature_query_id = athena_client.start_query_execution(
    QueryString=feature_snapshot_sql,
    QueryExecutionContext={
        "Database": FEATURE_DATABASE,
        "Catalog": "AwsDataCatalog"
    },
    ResultConfiguration={
        "OutputLocation": project_config["athena"]["query_output"]
    },
    WorkGroup=project_config["athena"]["workgroup"]
)["QueryExecutionId"]

print("Feature retrieval query:", feature_query_id)

for attempt in range(30):
    execution = athena_client.get_query_execution(
        QueryExecutionId=feature_query_id
    )["QueryExecution"]

    state = execution["Status"]["State"]
    if state == "SUCCEEDED":
        break
    if state in {"FAILED", "CANCELLED"}:
        raise RuntimeError(
            execution["Status"].get("StateChangeReason", state)
        )
    time.sleep(2)
else:
    raise TimeoutError(f"Query still running: {feature_query_id}")

# Load the actual Athena result from S3.
feature_result_uri = execution["ResultConfiguration"]["OutputLocation"]
location = urlparse(feature_result_uri)

response = s3_client.get_object(
    Bucket=location.netloc,
    Key=location.path.lstrip("/")
)
try:
    stored_features = pd.read_csv(
        response["Body"],
        dtype={"ticket_id": "string"},
        keep_default_na=False
    )
finally:
    response["Body"].close()

train_features_store = stored_features[
    stored_features["split"] == "train"
].copy()

validation_features_store = stored_features[
    stored_features["split"] == "validation"
].copy()

# Verify that retrieval preserved the original split membership.
assert stored_features["ticket_id"].is_unique
assert set(train_features_store["ticket_id"]) == set(train_df["ticket_id"])
assert set(validation_features_store["ticket_id"]) == set(
    validation_df["ticket_id"]
)

print("Training features retrieved:", len(train_features_store))
print("Validation features retrieved:", len(validation_features_store))
print("Verified: ticket IDs match the original splits.")

Feature retrieval query: 9b502988-5fcb-463b-9e72-b47cf7917d2c
Training features retrieved: 17710
Validation features retrieved: 4428
Verified: ticket IDs match the original splits.


In [12]:
import hashlib

feature_output_dir = PROJECT_ROOT / "data" / "features"
feature_output_dir.mkdir(parents=True, exist_ok=True)

feature_prefix = (
    f"features/{snapshot_id}/v1/"
    f"{int(feature_event_time * 1_000_000)}"
)

feature_datasets = {}

for name, retrieved, original in [
    ("train", train_features_store, train_features),
    ("validation", validation_features_store, validation_features),
]:
    # Verify that Feature Store preserved the feature values and labels.
    pd.testing.assert_frame_equal(
        retrieved[stored_columns]
        .sort_values("ticket_id").reset_index(drop=True),
        original[stored_columns]
        .sort_values("ticket_id").reset_index(drop=True),
        check_dtype=False
    )

    output_path = feature_output_dir / f"{name}.csv"
    retrieved[stored_columns].sort_values("ticket_id").to_csv(
        output_path, index=False
    )

    s3_key = f"{feature_prefix}/{name}.csv"
    s3_client.upload_file(str(output_path), PROJECT_BUCKET, s3_key)

    metadata = s3_client.head_object(
        Bucket=PROJECT_BUCKET, Key=s3_key
    )
    assert metadata["ContentLength"] == output_path.stat().st_size
    assert metadata.get("VersionId") not in (None, "null")

    feature_datasets[name] = {
        "uri": f"s3://{PROJECT_BUCKET}/{s3_key}",
        "version_id": metadata["VersionId"],
        "rows": len(retrieved),
        "sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
    }

    print(f"{name}: feature values verified; versioned file saved.")

project_config["feature_store"].update({
    "database": FEATURE_DATABASE,
    "table": FEATURE_TABLE,
    "retrieval_query_id": feature_query_id,
    "retrieval_result_uri": feature_result_uri,
    "datasets": feature_datasets,
    "text_column": TEXT_COLUMN,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Feature Store dataset locations saved in project_config.json.")

train: feature values verified; versioned file saved.
validation: feature values verified; versioned file saved.
Feature Store dataset locations saved in project_config.json.
